## Module 2: Disease Prediction System

### 1. Module Introduction and Objectives

This notebook implements, trains, persists, and validates **Module 2: Disease Prediction System** (`Disease_Predict_System.ipynb`), mapping patient symptom features to disease predictions and connecting with **Module 3 (Medicine Recommendation)**[cite: 1, 3, 18]:

1. **Data Loading & Sampling:** Loading `datasets/processed/Cleaned_Dataset_validated.csv`, creating a reproducible sample (`frac=0.2, random_state=42`), and separating symptom features (`X`) from target labels (`y` from `diseases`)[cite: 20, 24].
2. **Label Encoding & Train-Test Split:** Encoding disease labels using `LabelEncoder` (`disease_encoder.pkl`) and preparing data splits with `train_test_split`[cite: 18, 24].
3. **Model Training & Persistence:** Training the disease classification model and saving artifacts (`best_model.pkl` and `disease_encoder.pkl`) into `models/`[cite: 1, 18].
4. **End-to-End Integration Verification:** Testing symptom-based predictions with `get_recommendations_for_flask` across both matching and out-of-vocabulary disease classes[cite: 17, 18].

# Data Loading 

In [1]:
import pandas as pd
import joblib
import os

df = pd.read_csv('../datasets/processed/Cleaned_Dataset_validated.csv')
df_sampled = df.sample(frac=0.2, random_state=42)
X = df_sampled.drop(columns=['diseases'])
y = df_sampled['diseases']

# Processing

In [2]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Different ML Agorithms

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

models = {
    'Random Forest': RandomForestClassifier(),
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'K-Nearest Neighbors (KNN)': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB()
}

best_model = None
best_accuracy = 0
best_model_name = ""

# Model Training, Evaluate Metrics & Comparison 

In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("--- Training and Evaluating ML Models for Disease Prediction ---\n")
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    print(f"Model: {name}")
    print(f"  -> Accuracy  : {acc:.4f}")
    print(f"  -> Precision : {prec:.4f}")
    print(f"  -> Recall    : {rec:.4f}")
    print(f"  -> F1-Score  : {f1:.4f}\n")
    
    
    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model
        best_model_name = name

print(f"=== Best Model Selected: {best_model_name} (Accuracy: {best_accuracy:.4f}) ===")

--- Training and Evaluating ML Models for Disease Prediction ---

Model: Random Forest
  -> Accuracy  : 0.7944
  -> Precision : 0.7960
  -> Recall    : 0.7944
  -> F1-Score  : 0.7899

Model: Logistic Regression
  -> Accuracy  : 0.8236
  -> Precision : 0.8215
  -> Recall    : 0.8236
  -> F1-Score  : 0.8160

Model: Decision Tree
  -> Accuracy  : 0.7038
  -> Precision : 0.7197
  -> Recall    : 0.7038
  -> F1-Score  : 0.7038

Model: K-Nearest Neighbors (KNN)
  -> Accuracy  : 0.7411
  -> Precision : 0.7621
  -> Recall    : 0.7411
  -> F1-Score  : 0.7417

Model: Naive Bayes
  -> Accuracy  : 0.8404
  -> Precision : 0.8813
  -> Recall    : 0.8404
  -> F1-Score  : 0.8479

=== Best Model Selected: Naive Bayes (Accuracy: 0.8404) ===


# Model Persistence

In [5]:
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/best_model.pkl')
joblib.dump(encoder, '../models/disease_encoder.pkl')

print("Best model ('best_model.pkl') and disease encoder ('disease_encoder.pkl') successfully saved '../models/' folder!")

Best model ('best_model.pkl') and disease encoder ('disease_encoder.pkl') successfully saved '../models/' folder!


## End-to-End Integration Test (Module2 + Module3) 

In [13]:
import warnings
warnings.filterwarnings("ignore")

In [14]:
import sys
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
  sys.path.insert(0, str(REPO_ROOT))

from healthcare_webapp.services.flask_adapter import get_recommendations_for_flask

best_model = joblib.load(REPO_ROOT / "models" / "best_model.pkl")
encoder = joblib.load(REPO_ROOT / "models" / "disease_encoder.pkl")

if "X" in globals():
  feature_columns = X.columns
else:
  cleaned_df = pd.read_csv(
    REPO_ROOT / "datasets" / "processed" / "Cleaned_Dataset_validated.csv"
  )
  
  target_cols = [
      c for c in ["diseases", "Disease", "disease"] if c in cleaned_df.columns
  ]
  feature_columns = cleaned_df.drop(columns=target_cols).columns

def predict_disease(symptoms_list, feat_cols):
  cleaned_symptoms = [s.strip().lower() for s in symptoms_list]
  input_data = [
      1 if str(col).strip().lower() in cleaned_symptoms else 0
      for col in feat_cols
  ]
  input_array = np.array(input_data).reshape(1, -1)
  pred_encoded = best_model.predict(input_array)
  return encoder.inverse_transform(pred_encoded)[0]


def full_pipeline_test(
    symptoms_list, feat_cols, allergies=None, conditions=None, top_k=5
):
  predicted_disease = predict_disease(symptoms_list, feat_cols)
  flask_res = get_recommendations_for_flask(
      disease=predicted_disease,
      allergies=allergies,
      conditions=conditions,
      top_k=top_k,
      filter_allergies=True,
  )
  return predicted_disease, flask_res

sample_symptoms = ["anxiety", "depressio", "shortness_of_breath"]
disease_out, rec_out = full_pipeline_test(
    sample_symptoms, feature_columns, top_k=5
)

print("Predicted Disease:", disease_out)
print("Status:", rec_out["status"])
print("Count:", rec_out["count"])

if rec_out["count"] > 0:
  display(pd.DataFrame(rec_out["recommendations"]))
else:
  print("No medicines found; disclaimer:", rec_out.get("clinical_disclaimer"))


Predicted Disease: ascending cholangitis
Status: success
Count: 0
No medicines found; disclaimer: Academic research & educational prototype only. Recommendations and similarity scores do not establish clinical equivalence, pharmacological suitability, or a prescription. Absence of an allergen match does not guarantee clinical safety or absence of cross-reactivity. Always consult a qualified healthcare professional.


In [15]:
print("Total classes in encoder:", len(encoder.classes_))
print("Sample classes:", list(encoder.classes_[:15]))

non_empty_diseases = []
for d in encoder.classes_:
  res = get_recommendations_for_flask(disease=str(d), top_k=3)
  if res.get("count", 0) > 0:
    non_empty_diseases.append((d, res["count"]))

print(f"\nDiseases with medicines found ({len(non_empty_diseases)}):")
for d, cnt in non_empty_diseases[:15]:
  print(f" - {d} (count={cnt})")

Total classes in encoder: 686
Sample classes: ['abdominal aortic aneurysm', 'abdominal hernia', 'abscess of nose', 'abscess of the lung', 'abscess of the pharynx', 'acanthosis nigricans', 'acariasis', 'achalasia', 'acne', 'actinic keratosis', 'acute bronchiolitis', 'acute bronchitis', 'acute bronchospasm', 'acute glaucoma', 'acute kidney injury']

Diseases with medicines found (440):
 - abdominal aortic aneurysm (count=3)
 - abdominal hernia (count=3)
 - abscess of nose (count=3)
 - abscess of the lung (count=3)
 - acne (count=3)
 - actinic keratosis (count=2)
 - acute bronchiolitis (count=3)
 - acute bronchitis (count=3)
 - acute bronchospasm (count=3)
 - acute glaucoma (count=3)
 - acute kidney injury (count=3)
 - acute otitis media (count=3)
 - acute pancreatitis (count=3)
 - acute respiratory distress syndrome (ards) (count=3)
 - acute sinusitis (count=3)


In [16]:
if "cleaned_df" not in globals():
  possible_paths = [
      REPO_ROOT
      / "module1_data_collection_cleaning"
      / "Cleaned_Dataset.csv",
      REPO_ROOT / "datasets" / "processed" / "Cleaned_Dataset_validated.csv",
  ]
  for p in possible_paths:
    if p.exists():
      cleaned_df = pd.read_csv(p)
      break

target_col = (
    next(
        (
            c
            for c in ["diseases", "Disease", "disease"]
            if "cleaned_df" in globals() and c in cleaned_df.columns
        ),
        None,
    )
    if "cleaned_df" in globals()
    else None
)

chosen_disease = "acne"

if "cleaned_df" in globals() and target_col:
  matched = cleaned_df[
      cleaned_df[target_col].astype(str).str.strip().str.lower()
      == chosen_disease.lower()
  ]
  if not matched.empty:
    sym_means = matched[feature_columns].mean().sort_values(ascending=False)
    test_syms = sym_means[sym_means > 0].head(6).index.tolist()

    disease_out, rec_out = full_pipeline_test(
        test_syms, feature_columns, top_k=5
    )
    print("Test Symptoms:", test_syms)
    print("Predicted Disease:", disease_out)
    print("Status:", rec_out["status"])
    print("Count:", rec_out["count"])

    if rec_out["count"] > 0:
      display(pd.DataFrame(rec_out["recommendations"]))
  else:
    print(f"'{chosen_disease}' ke rows cleaned_df mein nahi mile.")
else:
  print(
      "cleaned_df nahi mila; aap manually feature_columns se symptoms pass"
      " karke test kar sakte hain."
  )

Test Symptoms: ['skin_dryness_peeling_scaliness_or_roughness', 'irregular_appearing_scalp', 'long_menstrual_periods', 'skin_rash', 'abnormal_appearing_skin', 'skin_swelling']
Predicted Disease: acne
Status: success
Count: 5


,medicine_name,composition,uses,side_effects,image_url,manufacturer,similarity_score,review_sentiment_score,experimental_ranking_score,ranking_methodology,match_explanation,safety_assessment,clinical_disclaimer
0,Apgel,Adapalene (0.1% w/w),Acne,Erythema skin redness Dry skin Skin peeling Sk...,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Hegde and Hegde Pharmaceutical LLP,0.3466,0.415,0.3569,85% TF-IDF Cosine Similarity + 15% Normalized ...,Retrieved because query matched indication key...,"{'allergy_detected': False, 'matched_allergens...",Academic research & educational prototype only...
1,Alight Plus Gel 15gm for Acne Treatment,Adapalene (0.1% w/w) + Clindamycin (1% w/w),Treatment of Acne,Skin peeling Erythema skin redness Itching Dry...,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Leeford Healthcare Ltd,0.3371,0.390,0.3450,85% TF-IDF Cosine Similarity + 15% Normalized ...,Retrieved because query matched indication key...,"{'allergy_detected': False, 'matched_allergens...",Academic research & educational prototype only...
2,Tretiva 5 Capsule,Isotretinoin (5mg),Treatment of Acne,Dry skin Increased liver enzymes Rash,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Intas Pharmaceuticals Ltd,0.3207,0.470,0.3431,85% TF-IDF Cosine Similarity + 15% Normalized ...,Retrieved because query matched indication key...,"{'allergy_detected': False, 'matched_allergens...",Academic research & educational prototype only...
3,Isopil 5 Tablet,Isotretinoin (5mg),Treatment of Acne,Dry skin Increased liver enzymes Rash,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Glowderma Labs Pvt Ltd,0.3192,0.610,0.3628,85% TF-IDF Cosine Similarity + 15% Normalized ...,Retrieved because query matched indication key...,"{'allergy_detected': False, 'matched_allergens...",Academic research & educational prototype only...
4,AB-Next Gel,Adapalene (0.1% w/w) + Benzoyl Peroxide (2.5% ...,Acne,Erythema skin redness Burning sensation Scalin...,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Lupin Ltd,0.3178,0.500,0.3451,85% TF-IDF Cosine Similarity + 15% Normalized ...,Retrieved because query matched indication key...,"{'allergy_detected': False, 'matched_allergens...",Academic research & educational prototype only...
